# 01b — Extract SoilGrids (ISRIC)
**Data source:** [SoilGrids REST API](https://rest.isric.org/) — 6 soil properties at 0-5cm depth

**Properties:** pH (H2O), Clay %, Sand %, Silt %, Organic Carbon Density, Cation Exchange Capacity

**Input:** `train_base.parquet`, `val_base.parquet` from notebook 00

**Output:** `soilgrids.parquet` (one row per unique station, 6 soil feature columns)

### API Resilience & Batch Optimizations
1. **Batching:** We bundle all 6 properties into a single REST API call per station, reducing calls from 972 to 162.
2. **Throttling:** We sleep for 6.0 seconds between stations to respect SoilGrids' strict API rate limit (max 5-10 requests/min).
3. **Exponential Backoff:** If the API fails or times out, we wait and retry with increasing delays.

**Estimated time:** ~15-20 min for ~162 stations (with throttling)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, time, requests, logging
import warnings
warnings.filterwarnings('ignore')

# === Logging Setup ===
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-5s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('01b_soilgrids')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold', 'font.size': 11})

INPUT_DIR  = '/kaggle/input/ey-water-quality-nb00'
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

LAT_COL     = 'Latitude'
LON_COL     = 'Longitude'
STATION_COL = 'station_id'

SOIL_PROPERTIES = ['phh2o', 'clay', 'sand', 'silt', 'ocd', 'cec']

log.info(f'Will extract {len(SOIL_PROPERTIES)} soil properties per station using batch API calls')

In [ ]:
# Helper function to find input file path in Kaggle
def find_file(filename, default_dir='/kaggle/working'):
    target = os.path.join(default_dir, filename)
    if os.path.exists(target):
        return target
    input_dir = '/kaggle/input'
    if os.path.exists(input_dir):
        for root, _, files in os.walk(input_dir):
            if filename in files:
                return os.path.join(root, filename)
    raise FileNotFoundError(f"File {filename} not found in input/working directories.")

train_base_path = find_file('train_base.parquet')
val_base_path   = find_file('val_base.parquet')

train_base = pd.read_parquet(train_base_path)
val_base   = pd.read_parquet(val_base_path)
all_data   = pd.concat([train_base, val_base], ignore_index=True)

unique_stations = all_data.groupby(STATION_COL)[[LAT_COL, LON_COL]].first().reset_index()
log.info(f'Unique stations: {len(unique_stations)}')

---
## Optimized Batch SoilGrids Query

In [ ]:
def fetch_soilgrids_batch(lat, lon, station_id, station_num, total_stations, retries=5):
    """
    Fetches all 6 soil properties in a single batch query (reduces calls by 6x).
    Uses custom headers and exponential backoff to handle rate limits and timeouts.
    """
    url = 'https://rest.isric.org/soilgrids/v2.0/properties/query'
    params = [
        ('lat', lat),
        ('lon', lon),
        ('depth', '0-5cm'),
        ('value', 'mean')
    ]
    for prop in SOIL_PROPERTIES:
        params.append(('property', prop))
        
    headers = {
        'User-Agent': 'curl/8.7.1',
        'Accept': '*/*'
    }
    
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=45)
            r.raise_for_status()
            data = r.json()
            layers = data.get('properties', {}).get('layers', [])
            
            results = {}
            for layer in layers:
                name = layer.get('name')
                depths = layer.get('depths', [])
                if depths:
                    val = depths[0].get('values', {}).get('mean')
                    results[f'soil_{name}'] = float(val) if val is not None else np.nan
                else:
                    results[f'soil_{name}'] = np.nan
                    
            if len(results) == len(SOIL_PROPERTIES):
                print(f"[{station_num:3d}/{total_stations}] {station_id[:20]:20s} -> Success (pH: {results.get('soil_phh2o', np.nan)/10:.1f})")
                return results
            else:
                raise ValueError(f"Expected {len(SOIL_PROPERTIES)} properties, got {len(results)}")
                
        except Exception as e:
            sleep_time = 6 * (attempt + 1)
            if attempt < retries - 1:
                log.warning(f"[{station_num}/{total_stations}] Query failed ({type(e).__name__}). Retrying in {sleep_time}s...")
                time.sleep(sleep_time)
            else:
                log.error(f"[{station_num}/{total_stations}] {station_id[:20]:20s} -> FAILED ALL RETRIES (Error: {type(e).__name__})")
                
    return {f'soil_{p}': np.nan for p in SOIL_PROPERTIES}

In [ ]:
total = len(unique_stations)
results = []
start_time = time.time()

log.info(f'Starting SoilGrids batch extraction for {total} stations...')

for idx, row in unique_stations.iterrows():
    lat, lon = row[LAT_COL], row[LON_COL]
    station = row[STATION_COL]
    
    # Single API request for all 6 properties
    soil_data = fetch_soilgrids_batch(lat, lon, station, idx + 1, total)
    
    record = {STATION_COL: station, LAT_COL: lat, LON_COL: lon}
    record.update(soil_data)
    results.append(record)
    
    # Sleep 6 seconds between stations to stay under 10 requests/min rate limit
    time.sleep(6.0)

elapsed_total = time.time() - start_time
log.info(f'DONE in {elapsed_total/60:.1f} min')

In [ ]:
soil_df = pd.DataFrame(results)

log.info(f'Output shape: {soil_df.shape}')
for prop in SOIL_PROPERTIES:
    col = f'soil_{prop}'
    nulls = soil_df[col].isnull().sum()
    log.info(f'  {col}: {nulls} nulls, '
             f'range [{soil_df[col].min():.1f}, {soil_df[col].max():.1f}]')

display(soil_df.describe())

---
## Figure: Soil Properties Map Grid

In [ ]:
n_props = len(SOIL_PROPERTIES)
ncols = 3
nrows = (n_props + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = axes.flatten()

for i, prop in enumerate(SOIL_PROPERTIES):
    ax = axes[i]
    col = f'soil_{prop}'
    valid = soil_df[col].notna()
    
    if valid.sum() > 0:
        sc = ax.scatter(soil_df.loc[valid, LON_COL], soil_df.loc[valid, LAT_COL],
                       c=soil_df.loc[valid, col], cmap='YlOrRd', s=50,
                       edgecolors='gray', linewidths=0.3)
        plt.colorbar(sc, ax=ax, shrink=0.8)
    
    if (~valid).any():
        ax.scatter(soil_df.loc[~valid, LON_COL], soil_df.loc[~valid, LAT_COL],
                  c='red', marker='x', s=60, linewidths=1.5)
    
    n_ok = valid.sum()
    ax.set_title(f'{prop} ({n_ok}/{len(soil_df)} OK)', fontsize=11)
    ax.set_xlim(16, 33); ax.set_ylim(-35, -22)
    ax.set_xlabel('Lon'); ax.set_ylabel('Lat')
    ax.grid(True, alpha=0.2)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('SoilGrids Properties per Station (0-5cm depth)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_01b_soilgrids_map.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Save Output

In [ ]:
out_path = f'{OUTPUT_DIR}/soilgrids.parquet'
soil_df.to_parquet(out_path, index=False)
size_kb = os.path.getsize(out_path) / 1024

log.info(f'Saved: {out_path} ({size_kb:.1f} KB, {len(soil_df)} rows, {len(SOIL_PROPERTIES)} properties)')
print(f'\n=== DONE ===')
print(f'Output: soilgrids.parquet')
print(f'Rows: {len(soil_df)}, Cols: {soil_df.columns.tolist()}')
print(f'Next: add this notebook output as dataset input for 01e')